# Taxi Rides

source: https://www.kaggle.com/datasets/yasserh/uber-fares-dataset/

## Nacitanie datasetu

In [1]:
import pandas as pd

url = 'https://bit.ly/4joetoh'
# url = 'https://github.com/bletvaska/pandas-101/raw/refs/heads/2023.12-tsys/data/uber.csv.gz'

# first time run - download and save dataset to file
# df = pd.read_csv(url, compression='gzip')
# df.to_csv('uber.csv')

# load dataset from local file
df = pd.read_csv('uber.csv', 
        # index_col=0, 
        header=0,
        names=['key', 'dt', 'price', 'pickup', 'pickup_lon', 'pickup_lat', 'dropoff_lon', 'dropoff_lat', 'passengers'],
        dtype={
                "passengers": "uint16",
                "price": "float16",
                "pickup_lon": "float16",
                "pickup_lat": "float16",
                "dropoff_lon": "float32",
                "dropoff_lat": "float32",
                "key": "uint32",
        },
        #parse_dates=['dt', 'pickup'],
        #date_format='%Y-%m-%d %H:%M:%S %Z',
)

# normalizing datetime columns
df['pickup'] = pd.to_datetime( 
    df["pickup"], 
    format="%Y-%m-%d %H:%M:%S UTC", 
    utc=True, 
).dt.as_unit("s")

df['dt'] = pd.to_datetime( 
    df["dt"], 
    format="ISO8601", 
    utc=True, 
).dt.as_unit("s")

### Gettings info about the table

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype             
---  ------       --------------   -----             
 0   key          200000 non-null  uint32            
 1   dt           200000 non-null  datetime64[s, UTC]
 2   price        200000 non-null  float16           
 3   pickup       200000 non-null  datetime64[s, UTC]
 4   pickup_lon   200000 non-null  float16           
 5   pickup_lat   200000 non-null  float16           
 6   dropoff_lon  199999 non-null  float32           
 7   dropoff_lat  199999 non-null  float32           
 8   passengers   200000 non-null  uint16            
dtypes: datetime64[s, UTC](2), float16(3), float32(2), uint16(1), uint32(1)
memory usage: 6.9 MB


In [3]:
type(df)
df

/home/mirek_binas/airflow/venv/lib/python3.12/site-packages/pandas/io/formats/format.py:1486: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,key,dt,price,pickup,pickup_lon,pickup_lat,dropoff_lon,dropoff_lat,passengers
0,24238194,2015-05-07 19:52:06+00:00,7.500000,2015-05-07 19:52:06+00:00,-74.0000,40.75000,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56+00:00,7.699219,2009-07-17 20:04:56+00:00,-74.0000,40.71875,-73.994713,40.750324,1
2,44984355,2009-08-24 21:45:00+00:00,12.898438,2009-08-24 21:45:00+00:00,-74.0000,40.75000,-73.962563,40.772648,1
3,25894730,2009-06-26 08:22:21+00:00,5.300781,2009-06-26 08:22:21+00:00,-74.0000,40.78125,-73.965317,40.803349,3
4,17610152,2014-08-28 17:47:00+00:00,16.000000,2014-08-28 17:47:00+00:00,-73.9375,40.75000,-73.973083,40.761246,5
...,...,...,...,...,...,...,...,...,...
199995,42598914,2012-10-28 10:49:00+00:00,3.000000,2012-10-28 10:49:00+00:00,-74.0000,40.75000,-73.986526,40.740295,1
199996,16382965,2014-03-14 01:09:00+00:00,7.500000,2014-03-14 01:09:00+00:00,-74.0000,40.75000,-74.006676,40.739620,1
199997,27804658,2009-06-29 00:42:00+00:00,30.906250,2009-06-29 00:42:00+00:00,-74.0000,40.75000,-73.858955,40.692589,2
199998,20259894,2015-05-20 14:56:25+00:00,14.500000,2015-05-20 14:56:25+00:00,-74.0000,40.71875,-73.983215,40.695415,1


In [4]:
df.describe()

/home/mirek_binas/airflow/venv/lib/python3.12/site-packages/pandas/core/nanops.py:1902: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
/home/mirek_binas/airflow/venv/lib/python3.12/site-packages/pandas/core/nanops.py:1902: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
/home/mirek_binas/airflow/venv/lib/python3.12/site-packages/pandas/core/nanops.py:1902: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)


,key,price,pickup_lon,pickup_lat,dropoff_lon,dropoff_lat,passengers
count,2.000000e+05,200000.000000,200000.000000,200000.000000,199999.000000,199999.000000,200000.000000
mean,2.771250e+07,11.360122,-72.532387,39.935698,-72.525291,39.923889,1.684535
std,1.601382e+07,0.000000,0.000000,0.000000,13.117408,6.794829,1.385997
min,1.000000e+00,-52.000000,-1341.000000,-74.000000,-3356.666260,-881.985535,0.000000
25%,1.382535e+07,6.000000,-74.000000,40.750000,-73.991409,40.733826,1.000000
50%,2.774550e+07,8.500000,-74.000000,40.750000,-73.980095,40.753040,1.000000
75%,4.155530e+07,12.500000,-73.937500,40.781250,-73.963661,40.768002,2.000000
max,5.542357e+07,499.000000,57.406250,1644.000000,1153.572632,872.697632,208.000000


In [5]:
df['pickup'] = pd.to_datetime( 
    df["pickup"], 
    format="%Y-%m-%d %H:%M:%S UTC", 
    utc=True, 
).dt.as_unit("s")

In [6]:
df['pickup']

0        2015-05-07 19:52:06+00:00
1        2009-07-17 20:04:56+00:00
2        2009-08-24 21:45:00+00:00
3        2009-06-26 08:22:21+00:00
4        2014-08-28 17:47:00+00:00
                    ...           
199995   2012-10-28 10:49:00+00:00
199996   2014-03-14 01:09:00+00:00
199997   2009-06-29 00:42:00+00:00
199998   2015-05-20 14:56:25+00:00
199999   2010-05-15 04:08:00+00:00
Name: pickup, Length: 200000, dtype: datetime64[s, UTC]

In [7]:
df.columns
df.dtypes
df.shape
df.size


1800000

In [8]:
'price' in df.columns

True

In [9]:
df.values

array([[24238194, Timestamp('2015-05-07 19:52:06+0000', tz='UTC'), 7.5,
        ..., -73.99951171875, 40.72321701049805, 1],
       [27835199, Timestamp('2009-07-17 20:04:56+0000', tz='UTC'),
        7.69921875, ..., -73.99471282958984, 40.75032424926758, 1],
       [44984355, Timestamp('2009-08-24 21:45:00+0000', tz='UTC'),
        12.8984375, ..., -73.96256256103516, 40.772647857666016, 1],
       ...,
       [27804658, Timestamp('2009-06-29 00:42:00+0000', tz='UTC'),
        30.90625, ..., -73.85895538330078, 40.692588806152344, 2],
       [20259894, Timestamp('2015-05-20 14:56:25+0000', tz='UTC'), 14.5,
        ..., -73.98321533203125, 40.69541549682617, 1],
       [11951496, Timestamp('2010-05-15 04:08:00+0000', tz='UTC'),
        14.1015625, ..., -73.98551177978516, 40.76879119873047, 1]],
      shape=(200000, 9), dtype=object)

In [10]:
type(df['price'])

pandas.Series